## How the data flow starting from the source code to the review ouput

Think of your project as a pipeline:
```
Python Code
    ↓
PythonParser
    ↓
AST Tree
    ↓
PythonASTAnalyzer
    ↓
ast_result (dictionary)
    ↓
┌───────────────────────────────────────┐
│              Rules                    │
│                                       │
│  unused imports                       │
│  unused variables                     │
│  unused functions                     │
│  complexity                           │
│  duplicate functions                  │
│  security                             │
└───────────────────────────────────────┘
    ↓
Findings
    ↓
Review Pipeline / Output
```

1. Parser creates the AST

Your parser takes:
```
def test():
    eval("hello")
```
and converts it into a Python AST tree.

2. `Python_ast_analyzer` examines the AST

The analyzer walks through that tree:
```
for node in ast.walk(tree):
```
and extracts useful information.

For example:
```
{
    "functions": [
        {"name": "test", "line": 1}
    ],


    "function_calls": [
        {"name": "eval", "line": 2}
    ],


    "imports": [],
    "variables": [],
    "loops": [],
    "conditions": [],
    "name_usage": []
}
```
### This dictionary is the data produced by the analyzer.
---
3. Rules receive that dictionary

Your security rule doesn't analyze the original Python code or AST directly.

It receives:

`ast_result`

For example:

`def check_security(ast_result):`

Then it looks at:

`ast_result["function_calls"]`

and finds:

`{"name": "eval", "line": 2}`

Then it creates a `Finding` object.

4. Each rule uses different parts

For example:
```
ast_result
│
├── functions ───────→ unused functions
│                       complexity
│                       duplicate functions
│
├── imports ─────────→ unused imports
│
├── variables ───────→ unused variables
│
├── function_calls ──→ security checks
│
├── loops ───────────→ complexity
│
└── conditions ──────→ complexity
```
So **the analyzer is responsible for collecting information**, while **the rules are responsible for deciding whether that information represents a problem**.


